# Case Linking — Link Generation Tests

Verifies the three gold tables produced by `GOLD_ACTIVE_CASE_LINKING.ipynb`:
- `aria_ccd_case_mappings` — ARIA CaseNo → CCD reference
- `case_link_combinations` — directional pairs with CCD refs and reason lookup
- `case_link_payloads` — CCD-shaped JSON payload per "from" case

In [0]:
########################
# test Setup
########################

state_under_test = "caseLinking"
run_tag_default = "link_generation"

dbutils.widgets.text("run_tag", run_tag_default)
run_tag = dbutils.widgets.get("run_tag") or run_tag_default

In [0]:
import os
import sys
import re
import time as perf_time
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, row_number, count, size, array_distinct
from pyspark.sql.window import Window
import inspect

run_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
test_results_path = "/Workspace/Users/" + run_user + "/Results/Case_Linking_Tests"
os.makedirs(test_results_path, exist_ok=True)

for _mod in list(sys.modules.keys()):
    if _mod.startswith("Test_Functions") or _mod == "models.test_result":
        del sys.modules[_mod]

from models.test_result import TestResult
from Test_Functions.report_helpers import (
    build_report_folder,
    write_csv, write_xlsx, write_html,
    count_by_status,
    create_run, create_results,
)

print(f"User: {run_user}")
print(f"state_under_test: {state_under_test}")
print(f"run_tag: {run_tag}")
print(f"Results folder root: {test_results_path}")

In [0]:
#######################
# Spark Config (state-independent — runs once)
#######################
config_path = "dbfs:/configs/config.json"
config = spark.read.option("multiline", "true").json(config_path)
first_row = config.first()
env = first_row["env"].strip().lower()
env_name = env
lz_key = first_row["lz_key"].strip().lower()
keyvault_name = f"ingest{lz_key}-meta002-{env}"
KeyVault_name = keyvault_name

client_secret = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-TENANT-ID")
client_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-ID")

# Storage accounts + spark.conf
curated_storage    = f"ingest{lz_key}curated{env}"
checkpoint_storage = f"ingest{lz_key}xcutting{env}"
raw_storage        = f"ingest{lz_key}raw{env}"
landing_storage    = f"ingest{lz_key}landing{env}"
external_storage   = f"ingest{lz_key}external{env}"

for storage in (curated_storage, checkpoint_storage, raw_storage, landing_storage, external_storage):
    spark.conf.set(f"fs.azure.account.auth.type.{storage}.dfs.core.windows.net", "OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage}.dfs.core.windows.net", client_id)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage}.dfs.core.windows.net", client_secret)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

bronze_path = f"abfss://bronze@ingest{lz_key}curated{env}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"
silver_path = f"abfss://silver@ingest{lz_key}curated{env}.dfs.core.windows.net/ARIADM/ACTIVE/CCD/APPEALS/"

print(f"env={env}  lz_key={lz_key}")

## Load tables

In [0]:
aria_ccd_case_mappings           = spark.table("hive_metastore.ariadm_active_appeals.aria_ccd_case_mappings")
case_link_combinations           = spark.table("hive_metastore.ariadm_active_appeals.case_link_combinations")
case_link_payloads               = spark.table("hive_metastore.ariadm_active_appeals.case_link_payloads")
silver_case_link_groups          = spark.table("hive_metastore.ariadm_active_appeals.silver_case_link_groups")
silver_case_link_reason_mappings = spark.table("hive_metastore.ariadm_active_appeals.silver_case_link_reason_mappings")

print(f"aria_ccd_case_mappings rows:           {aria_ccd_case_mappings.count():,}")
print(f"case_link_combinations rows:           {case_link_combinations.count():,}")
print(f"case_link_payloads rows:               {case_link_payloads.count():,}")
print(f"silver_case_link_groups rows:          {silver_case_link_groups.count():,}")
print(f"silver_case_link_reason_mappings rows: {silver_case_link_reason_mappings.count():,}")

In [0]:
run_start_datetime = datetime.utcnow()
all_test_results = []
t0 = perf_time.time()

# Test aria_ccd_case_mappings

In [0]:
def test_aria_ccd_quality(df):
    fn = inspect.stack()[0].function
    results = []
    try:
        total = df.count()

        # No nulls
        null_caseno = df.filter(col("CaseNo").isNull()).count()
        null_ccdid  = df.filter(col("CCDCaseID").isNull()).count()
        ok = null_caseno == 0 and null_ccdid == 0
        results.append(TestResult(
            "aria_ccd_case_mappings.no_nulls",
            "PASS" if ok else "FAIL",
            f"expected: 0/0 null (CaseNo/CCDCaseID) | actual: {null_caseno}/{null_ccdid}",
            state_under_test, fn))

        # CaseNo uniqueness
        unique = df.select("CaseNo").distinct().count()
        ok = unique == total
        msg = f"expected: {total} distinct CaseNo | actual: {unique}"
        if not ok:
            dupes = df.groupBy("CaseNo").count().filter("count > 1").limit(5).collect()
            msg += f" (sample dupes: {[(r.CaseNo, r['count']) for r in dupes]})"
        results.append(TestResult(
            "aria_ccd_case_mappings.caseno_unique",
            "PASS" if ok else "FAIL",
            msg, state_under_test, fn))

        # CaseNo format XX/NNNNN/NNNN
        bad_caseno_df = df.filter(~col("CaseNo").rlike(r"^[A-Z]{2}/\d{5}/\d{4}$"))
        bad_n = bad_caseno_df.count()
        msg = f"expected: all match XX/NNNNN/NNNN | actual: {bad_n} bad of {total}"
        if bad_n:
            sample = [r.CaseNo for r in bad_caseno_df.select("CaseNo").limit(5).collect()]
            msg += f" (sample: {sample})"
        results.append(TestResult(
            "aria_ccd_case_mappings.caseno_format",
            "PASS" if bad_n == 0 else "FAIL",
            msg, state_under_test, fn))

        # CCDCaseID format: 16 digits
        bad_ccd_df = df.filter(~col("CCDCaseID").rlike(r"^\d{16}$"))
        bad_n = bad_ccd_df.count()
        msg = f"expected: 16 digits | actual: {bad_n} bad of {total}"
        if bad_n:
            sample = [r.CCDCaseID for r in bad_ccd_df.select("CCDCaseID").limit(5).collect()]
            msg += f" (sample: {sample})"
        results.append(TestResult(
            "aria_ccd_case_mappings.ccdcaseid_format",
            "PASS" if bad_n == 0 else "FAIL",
            msg, state_under_test, fn))
    except Exception as e:
        results.append(TestResult(
            "aria_ccd_case_mappings", "FAIL", f"Test crashed: {e}",
            state_under_test, fn))
    return results


all_test_results.extend(test_aria_ccd_quality(aria_ccd_case_mappings))

# Test case_link_combinations

In [0]:
def test_combinations_count_formula(combinations_df, groups_df):
    """Approach A — first-principles count using n(n-1)/2 per LinkNo.

    For each LinkNo group with n distinct cases, the LLD says n(n-1)/2 pairs.
    Sum across all LinkNos and compare to the combinations row count. Independent
    of the dev's row_number + self-join approach — pure arithmetic on case counts."""
    fn = inspect.stack()[0].function
    results = []
    try:
        per_linkno = (
            groups_df.filter(col("LinkNo").isNotNull())
                     .groupBy("LinkNo")
                     .agg(F.countDistinct("CaseNo").alias("n"))
        )
        per_linkno = per_linkno.withColumn(
            "expected_pairs",
            (col("n") * (col("n") - 1) / F.lit(2)).cast("long")
        )

        expected_total = per_linkno.agg(F.sum("expected_pairs").alias("t")).first().t or 0
        actual_total   = combinations_df.count()

        ok = expected_total == actual_total
        msg = f"expected: {expected_total} pairs (sum of n(n-1)/2 per LinkNo) | actual: {actual_total} rows in case_link_combinations"
        if not ok:
            # Show a sample of LinkNo groups for context
            top_groups = per_linkno.orderBy(col("expected_pairs").desc()).limit(5).collect()
            sample = [(r.LinkNo, r.n, r.expected_pairs) for r in top_groups]
            msg += f" (largest groups LinkNo,n,expected_pairs: {sample})"
        results.append(TestResult(
            "case_link_combinations.count_formula",
            "PASS" if ok else "FAIL", msg, state_under_test, fn))
    except Exception as e:
        results.append(TestResult(
            "case_link_combinations.count_formula", "FAIL", f"Test crashed: {e}",
            state_under_test, fn))
    return results


all_test_results.extend(test_combinations_count_formula(case_link_combinations, silver_case_link_groups))

In [0]:
def test_combinations_match_itertools(combinations_df, groups_df):
    """Approach D — derive expected (From, To, ReasonLinkId) triples in pure Python
    via itertools.combinations, then compare to combinations. Uses no Spark window,
    no Spark self-join — mechanically distinct from the dev implementation."""
    import itertools
    from collections import defaultdict

    fn = inspect.stack()[0].function
    results = []
    try:
        # Pull silver to driver (small data set).
        silver_rows = (
            groups_df.filter(col("LinkNo").isNotNull())
                     .select("CaseNo", "BirthDate", "LinkNo", "ReasonLinkId")
                     .collect()
        )

        by_linkno = defaultdict(list)
        for r in silver_rows:
            by_linkno[r.LinkNo].append((r.CaseNo, r.BirthDate, r.ReasonLinkId))

        # For each LinkNo: sort by (BirthDate ASC, CaseNo ASC), build i<j pairs.
        # The "from" of each pair is the earlier-sorted case; ReasonLinkId comes from "from".
        expected_set = set()
        for linkno, members in by_linkno.items():
            members_sorted = sorted(members, key=lambda x: (x[1] is not None, x[1].isoformat() if x[1] is not None else "", x[0]))  # NULLS FIRST, matches Spark orderBy
            for a, b in itertools.combinations(members_sorted, 2):
                expected_set.add((a[0], b[0], a[2]))

        # Pull actual to driver
        actual_rows = (
            combinations_df.select("CaseNoFrom", "CaseNoTo", "ReasonLinkId").collect()
        )
        actual_set = set((r.CaseNoFrom, r.CaseNoTo, r.ReasonLinkId) for r in actual_rows)

        missing = expected_set - actual_set
        extra   = actual_set - expected_set

        ok = not missing and not extra
        msg = (f"expected: {len(expected_set)} distinct triples (itertools.combinations) | "
               f"actual: {len(actual_set)} distinct triples")
        if missing:
            sample = list(missing)[:5]
            msg += f"; {len(missing)} missing (sample: {sample})"
        if extra:
            sample = list(extra)[:5]
            msg += f"; {len(extra)} unexpected (sample: {sample})"
        results.append(TestResult(
            "case_link_combinations.match_itertools",
            "PASS" if ok else "FAIL", msg, state_under_test, fn))
    except Exception as e:
        results.append(TestResult(
            "case_link_combinations.match_itertools", "FAIL", f"Test crashed: {e}",
            state_under_test, fn))
    return results


all_test_results.extend(test_combinations_match_itertools(case_link_combinations, silver_case_link_groups))

In [0]:
def test_combinations_no_self_or_reverse(combinations_df):
    fn = inspect.stack()[0].function
    results = []
    try:
        # No self-pair
        self_df = combinations_df.filter(col("CaseNoFrom") == col("CaseNoTo"))
        self_n = self_df.count()
        msg = f"expected: 0 self-pairs | actual: {self_n}"
        if self_n:
            sample = [r.CaseNoFrom for r in self_df.select("CaseNoFrom").limit(5).collect()]
            msg += f" (sample: {sample})"
        results.append(TestResult(
            "case_link_combinations.no_self_pairs",
            "PASS" if self_n == 0 else "FAIL", msg, state_under_test, fn))

        # No reverse pairs: if (A,B) exists, (B,A) must not also exist.
        # Filter out self-pairs (they trivially match themselves) and keep
        # only one direction of any (A,B) <-> (B,A) by requiring c1.from < c1.to.
        non_self = combinations_df.filter(col("CaseNoFrom") != col("CaseNoTo"))
        rev = (
            non_self.alias("c1")
                .join(non_self.alias("c2"),
                      (col("c1.CaseNoFrom") == col("c2.CaseNoTo")) &
                      (col("c1.CaseNoTo")   == col("c2.CaseNoFrom")) &
                      (col("c1.CaseNoFrom") <  col("c1.CaseNoTo")),
                      "inner")
                .select(
                    col("c1.CaseNoFrom").alias("from_a"),
                    col("c1.CaseNoTo").alias("to_a"),
                )
                .distinct()
        )
        rev_n = rev.count()
        msg = f"expected: 0 reverse pairs | actual: {rev_n}"
        if rev_n:
            sample = [(r.from_a, r.to_a) for r in rev.limit(5).collect()]
            msg += f" (sample: {sample})"
        results.append(TestResult(
            "case_link_combinations.no_reverse_pairs",
            "PASS" if rev_n == 0 else "FAIL", msg, state_under_test, fn))
    except Exception as e:
        results.append(TestResult(
            "case_link_combinations.self_reverse", "FAIL", f"Test crashed: {e}",
            state_under_test, fn))
    return results


all_test_results.extend(test_combinations_no_self_or_reverse(case_link_combinations))

In [0]:
def test_combinations_key_description_match(combinations_df, mappings_df):
    """Every row's (key, description) must equal the silver mapping for that ReasonLinkId."""
    fn = inspect.stack()[0].function
    results = []
    try:
        joined = combinations_df.alias("c").join(
            mappings_df.alias("m"), col("c.ReasonLinkId") == col("m.ReasonLinkId"), "left"
        )

        # null on mapping side = ReasonLinkId not in silver mapping at all
        no_mapping = joined.filter(col("m.ReasonLinkId").isNull())
        no_mapping_n = no_mapping.count()

        # key mismatch
        bad_key = joined.filter(
            col("m.ReasonLinkId").isNotNull() & (col("c.key") != col("m.key"))
        )
        bad_key_n = bad_key.count()

        # description mismatch (normalise empty <-> null)
        def norm(c):
            return F.when(c.isNull() | (c == ""), F.lit(None)).otherwise(c)

        bad_desc = joined.filter(
            col("m.ReasonLinkId").isNotNull() &
            (norm(col("c.description")) != norm(col("m.description"))) &
            ~(norm(col("c.description")).isNull() & norm(col("m.description")).isNull())
        )
        bad_desc_n = bad_desc.count()

        ok = no_mapping_n == 0 and bad_key_n == 0 and bad_desc_n == 0
        parts = [f"unmapped ReasonLinkId={no_mapping_n}",
                 f"key mismatches={bad_key_n}",
                 f"description mismatches={bad_desc_n}"]
        results.append(TestResult(
            "case_link_combinations.key_description_match",
            "PASS" if ok else "FAIL",
            f"expected: 0/0/0 | actual: " + ", ".join(parts),
            state_under_test, fn))
    except Exception as e:
        results.append(TestResult(
            "case_link_combinations.key_description_match", "FAIL", f"Test crashed: {e}",
            state_under_test, fn))
    return results


all_test_results.extend(test_combinations_key_description_match(case_link_combinations, silver_case_link_reason_mappings))

In [0]:
def test_combinations_ccd_enrichment(combinations_df, mappings_df):
    """If an ARIA CaseNo is in aria_ccd_case_mappings, its CCD ref on that side
    of the combinations row must be populated and equal."""
    fn = inspect.stack()[0].function
    results = []
    try:
        # FROM side
        from_join = combinations_df.alias("c").join(
            mappings_df.alias("m"), col("c.CaseNoFrom") == col("m.CaseNo"), "left"
        )
        from_bad = from_join.filter(
            col("m.CaseNo").isNotNull() & (
                col("c.CCDCaseReferenceNumberFrom").isNull() |
                (col("c.CCDCaseReferenceNumberFrom") != col("m.CCDCaseID"))
            )
        )
        from_bad_n = from_bad.count()
        msg = f"expected: CCDCaseReferenceNumberFrom matches mapping where ARIA case exists | actual: {from_bad_n} mismatches"
        if from_bad_n:
            sample = [(r["c.CaseNoFrom"], r["c.CCDCaseReferenceNumberFrom"], r["m.CCDCaseID"])
                      for r in from_bad.select("c.CaseNoFrom", "c.CCDCaseReferenceNumberFrom", "m.CCDCaseID").limit(5).collect()]
            msg += f" (sample: {sample})"
        results.append(TestResult(
            "case_link_combinations.ccd_ref_from",
            "PASS" if from_bad_n == 0 else "FAIL", msg, state_under_test, fn))

        # TO side
        to_join = combinations_df.alias("c").join(
            mappings_df.alias("m"), col("c.CaseNoTo") == col("m.CaseNo"), "left"
        )
        to_bad = to_join.filter(
            col("m.CaseNo").isNotNull() & (
                col("c.CCDCaseReferenceNumberTo").isNull() |
                (col("c.CCDCaseReferenceNumberTo") != col("m.CCDCaseID"))
            )
        )
        to_bad_n = to_bad.count()
        msg = f"expected: CCDCaseReferenceNumberTo matches mapping where ARIA case exists | actual: {to_bad_n} mismatches"
        if to_bad_n:
            sample = [(r["c.CaseNoTo"], r["c.CCDCaseReferenceNumberTo"], r["m.CCDCaseID"])
                      for r in to_bad.select("c.CaseNoTo", "c.CCDCaseReferenceNumberTo", "m.CCDCaseID").limit(5).collect()]
            msg += f" (sample: {sample})"
        results.append(TestResult(
            "case_link_combinations.ccd_ref_to",
            "PASS" if to_bad_n == 0 else "FAIL", msg, state_under_test, fn))
    except Exception as e:
        results.append(TestResult(
            "case_link_combinations.ccd_enrichment", "FAIL", f"Test crashed: {e}",
            state_under_test, fn))
    return results


all_test_results.extend(test_combinations_ccd_enrichment(case_link_combinations, aria_ccd_case_mappings))

In [0]:
def test_combinations_caseno_format(combinations_df):
    fn = inspect.stack()[0].function
    results = []
    try:
        pattern = r"^[A-Z]{2}/\d{5}/\d{4}$"
        bad_from_df = combinations_df.filter(~col("CaseNoFrom").rlike(pattern))
        bad_to_df   = combinations_df.filter(~col("CaseNoTo").rlike(pattern))
        bf = bad_from_df.count()
        bt = bad_to_df.count()
        ok = bf == 0 and bt == 0
        msg = f"expected: all match XX/NNNNN/NNNN | actual: {bf} bad CaseNoFrom, {bt} bad CaseNoTo"
        if bf:
            msg += f" (from sample: {[r.CaseNoFrom for r in bad_from_df.select('CaseNoFrom').limit(5).collect()]})"
        if bt:
            msg += f" (to sample: {[r.CaseNoTo for r in bad_to_df.select('CaseNoTo').limit(5).collect()]})"
        results.append(TestResult(
            "case_link_combinations.caseno_format",
            "PASS" if ok else "FAIL", msg, state_under_test, fn))
    except Exception as e:
        results.append(TestResult(
            "case_link_combinations.caseno_format", "FAIL", f"Test crashed: {e}",
            state_under_test, fn))
    return results


all_test_results.extend(test_combinations_caseno_format(case_link_combinations))

# Test case_link_payloads

In [0]:
def test_payloads_from_uniqueness(payloads_df):
    fn = inspect.stack()[0].function
    results = []
    try:
        total = payloads_df.count()
        distinct = payloads_df.select("CCDCaseReferenceNumberFrom").distinct().count()
        ok = total == distinct
        msg = f"expected: {total} distinct CCDCaseReferenceNumberFrom | actual: {distinct}"
        if not ok:
            dupes = payloads_df.groupBy("CCDCaseReferenceNumberFrom").count().filter("count > 1").limit(5).collect()
            msg += f" (sample: {[(r.CCDCaseReferenceNumberFrom, r['count']) for r in dupes]})"
        results.append(TestResult(
            "case_link_payloads.from_unique",
            "PASS" if ok else "FAIL", msg, state_under_test, fn))
    except Exception as e:
        results.append(TestResult(
            "case_link_payloads.from_unique", "FAIL", f"Test crashed: {e}",
            state_under_test, fn))
    return results


all_test_results.extend(test_payloads_from_uniqueness(case_link_payloads))

In [0]:
def test_payloads_caselinks_count(payloads_df, combinations_df):
    """caseLinks length for each From == count of fully-resolved combinations for that From."""
    fn = inspect.stack()[0].function
    results = []
    try:
        expected_counts = (
            combinations_df
                .filter(col("CCDCaseReferenceNumberFrom").isNotNull() &
                        col("CCDCaseReferenceNumberTo").isNotNull())
                .groupBy("CCDCaseReferenceNumberFrom").agg(count("*").alias("expected_n"))
        )

        actual_counts = payloads_df.select(
            "CCDCaseReferenceNumberFrom",
            size("caseLinks").alias("actual_n"),
        )

        joined = expected_counts.join(actual_counts, "CCDCaseReferenceNumberFrom", "outer") \
                                .na.fill({"expected_n": 0, "actual_n": 0})

        bad = joined.filter(col("expected_n") != col("actual_n"))
        bad_n = bad.count()
        msg = f"expected: caseLinks length == combinations count per From | actual: {bad_n} mismatches"
        if bad_n:
            sample = [(r.CCDCaseReferenceNumberFrom, r.expected_n, r.actual_n)
                      for r in bad.limit(5).collect()]
            msg += f" (sample From,expected,actual: {sample})"
        results.append(TestResult(
            "case_link_payloads.caselinks_count",
            "PASS" if bad_n == 0 else "FAIL", msg, state_under_test, fn))
    except Exception as e:
        results.append(TestResult(
            "case_link_payloads.caselinks_count", "FAIL", f"Test crashed: {e}",
            state_under_test, fn))
    return results


all_test_results.extend(test_payloads_caselinks_count(case_link_payloads, case_link_combinations))

In [0]:
def test_payloads_entry_structure(payloads_df):
    """For every caseLinks entry, check: id == CaseReference, CaseType == 'Asylum',
    ReasonForLink.id is a UUID v4, and CreatedDateTime is ISO8601 to milliseconds."""
    fn = inspect.stack()[0].function
    results = []
    try:
        exploded = (
            payloads_df
                .select("CCDCaseReferenceNumberFrom", F.explode("caseLinks").alias("link"))
                .select(
                    "CCDCaseReferenceNumberFrom",
                    col("link.id").alias("link_id"),
                    col("link.value.CaseType").alias("CaseType"),
                    col("link.value.CaseReference").alias("CaseReference"),
                    col("link.value.ReasonForLink").alias("ReasonForLink"),
                    col("link.value.CreatedDateTime").alias("CreatedDateTime"),
                )
        )

        total = exploded.count()

        # id == CaseReference
        bad_id_df = exploded.filter(col("link_id") != col("CaseReference"))
        bad_id_n = bad_id_df.count()
        msg = f"expected: id == CaseReference | actual: {bad_id_n} of {total} mismatches"
        if bad_id_n:
            sample = [(r.link_id, r.CaseReference) for r in bad_id_df.select("link_id", "CaseReference").limit(5).collect()]
            msg += f" (sample: {sample})"
        results.append(TestResult(
            "case_link_payloads.entry_id_eq_caseref",
            "PASS" if bad_id_n == 0 else "FAIL", msg, state_under_test, fn))

        # CaseType always "Asylum"
        bad_ct_df = exploded.filter(col("CaseType") != F.lit("Asylum"))
        bad_ct_n = bad_ct_df.count()
        msg = f"expected: CaseType == 'Asylum' | actual: {bad_ct_n} of {total} mismatches"
        if bad_ct_n:
            sample = [r.CaseType for r in bad_ct_df.select("CaseType").distinct().limit(5).collect()]
            msg += f" (distinct bad values: {sample})"
        results.append(TestResult(
            "case_link_payloads.entry_casetype",
            "PASS" if bad_ct_n == 0 else "FAIL", msg, state_under_test, fn))

        # ReasonForLink.id is a UUID v4
        rfl = exploded.select("link_id", F.explode("ReasonForLink").alias("rfl")) \
                       .select(col("rfl.id").alias("rfl_id"))
        total_rfl = rfl.count()
        uuid_pattern = r"^[0-9a-f]{8}-[0-9a-f]{4}-4[0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}$"
        bad_uuid_df = rfl.filter(~col("rfl_id").rlike(uuid_pattern))
        bad_uuid_n = bad_uuid_df.count()
        msg = f"expected: UUID v4 format | actual: {bad_uuid_n} of {total_rfl} bad"
        if bad_uuid_n:
            sample = [r.rfl_id for r in bad_uuid_df.limit(5).collect()]
            msg += f" (sample: {sample})"
        results.append(TestResult(
            "case_link_payloads.entry_reasonforlink_uuid",
            "PASS" if bad_uuid_n == 0 else "FAIL", msg, state_under_test, fn))

        # CreatedDateTime ISO8601 to milliseconds: yyyy-MM-ddTHH:mm:ss.SSS
        dt_pattern = r"^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d{3}$"
        bad_dt_df = exploded.filter(~col("CreatedDateTime").rlike(dt_pattern))
        bad_dt_n = bad_dt_df.count()
        msg = f"expected: yyyy-MM-ddTHH:mm:ss.SSS | actual: {bad_dt_n} of {total} bad"
        if bad_dt_n:
            sample = [r.CreatedDateTime for r in bad_dt_df.select("CreatedDateTime").limit(5).collect()]
            msg += f" (sample: {sample})"
        results.append(TestResult(
            "case_link_payloads.entry_createddatetime",
            "PASS" if bad_dt_n == 0 else "FAIL", msg, state_under_test, fn))
    except Exception as e:
        results.append(TestResult(
            "case_link_payloads.entry_structure", "FAIL", f"Test crashed: {e}",
            state_under_test, fn))
    return results


all_test_results.extend(test_payloads_entry_structure(case_link_payloads))

In [0]:
def test_payloads_top_level_id_unique(payloads_df):
    """Within each caseLinks array, the top-level 'id' must be unique.
    Duplicates cause CCD validate to return 422 — the known same-case-in-multiple-groups issue."""
    fn = inspect.stack()[0].function
    results = []
    try:
        with_counts = payloads_df.withColumn(
            "link_count", size("caseLinks")
        ).withColumn(
            "distinct_id_count",
            size(array_distinct(expr("transform(caseLinks, x -> x.id)")))
        )

        dupes_df = with_counts.filter(col("link_count") != col("distinct_id_count"))
        dupes_n = dupes_df.count()
        msg = f"expected: 0 payloads with duplicate top-level ids | actual: {dupes_n}"
        if dupes_n:
            sample = [(r.CCDCaseReferenceNumberFrom, r.link_count, r.distinct_id_count)
                      for r in dupes_df.select("CCDCaseReferenceNumberFrom", "link_count", "distinct_id_count").limit(5).collect()]
            msg += f" (sample From,total,distinct: {sample})"
        results.append(TestResult(
            "case_link_payloads.top_level_id_unique",
            "PASS" if dupes_n == 0 else "FAIL", msg, state_under_test, fn))
    except Exception as e:
        results.append(TestResult(
            "case_link_payloads.top_level_id_unique", "FAIL", f"Test crashed: {e}",
            state_under_test, fn))
    return results


all_test_results.extend(test_payloads_top_level_id_unique(case_link_payloads))

In [0]:
def test_payloads_other_description_policy(payloads_df):
    """OtherDescription should be populated ONLY when Reason == 'CLRC014'.
    For other reasons it must be null or empty (per the demo agreement)."""
    fn = inspect.stack()[0].function
    results = []
    try:
        exploded = (
            payloads_df
                .select(F.explode("caseLinks").alias("link"))
                .select(F.explode("link.value.ReasonForLink").alias("rfl"))
                .select(
                    col("rfl.value.Reason").alias("Reason"),
                    col("rfl.value.OtherDescription").alias("OtherDescription"),
                )
        )

        # CLRC014 must have non-empty OtherDescription
        clrc014_missing = exploded.filter(
            (col("Reason") == "CLRC014") &
            (col("OtherDescription").isNull() | (col("OtherDescription") == ""))
        )
        cm_n = clrc014_missing.count()

        # Non-CLRC014 must have empty/null OtherDescription
        non_clrc014_set = exploded.filter(
            (col("Reason") != "CLRC014") &
            col("OtherDescription").isNotNull() & (col("OtherDescription") != "")
        )
        nc_n = non_clrc014_set.count()

        ok = cm_n == 0 and nc_n == 0
        parts = [f"CLRC014 with empty OtherDescription={cm_n}",
                 f"non-CLRC014 with populated OtherDescription={nc_n}"]
        msg = f"expected: 0/0 | actual: " + ", ".join(parts)
        if nc_n:
            sample = [(r.Reason, r.OtherDescription)
                      for r in non_clrc014_set.select("Reason", "OtherDescription").distinct().limit(5).collect()]
            msg += f" (non-CLRC014 sample Reason,OtherDescription: {sample})"
        results.append(TestResult(
            "case_link_payloads.other_description_policy",
            "PASS" if ok else "FAIL", msg, state_under_test, fn))
    except Exception as e:
        results.append(TestResult(
            "case_link_payloads.other_description_policy", "FAIL", f"Test crashed: {e}",
            state_under_test, fn))
    return results


all_test_results.extend(test_payloads_other_description_policy(case_link_payloads))

In [0]:
elapsed = perf_time.time() - t0
counts = count_by_status(all_test_results)

print(f"Tests complete in {elapsed:.1f}s")
print(f"  Total : {counts['TOTAL']}")
print(f"  Pass  : {counts['PASS']}")
print(f"  Fail  : {counts['FAIL']}")
print(f"  NoData: {counts['NO_DATA']}")
print(f"  Error : {counts['ERROR']}")

## Write reports + audit

In [0]:
# Sort fails to the top, then alphabetical by test_field within each status bucket.
status_order = {"FAIL": 0, "ERROR": 1, "NO_DATA": 2, "PASS": 3}
all_test_results.sort(key=lambda r: (status_order.get(str(r.status).upper(), 4), str(r.test_field)))

folder, timestamp, _ = build_report_folder(test_results_path, f"{state_under_test}_{run_tag}")
print(f"Reports folder: {folder}")

try:
    csv_path  = write_csv(all_test_results, state_under_test, folder, timestamp)
    print(f"CSV  : {csv_path}")
except Exception as e:
    print(f"write_csv FAILED: {e}")

try:
    run_id = create_run(
        spark,
        run_user=run_user,
        run_start_datetime=run_start_datetime,
        run_by_automation_name="Case_Linking_Tests",
        run_tag=run_tag,
        run_status=("FAILED" if any(str(getattr(r, "status", "")).upper().strip() in ("FAIL", "ERROR") for r in all_test_results) else "PASSED"),
        state_under_test=state_under_test,
    )
    print(f"Created run_id={run_id}")
    n = create_results(spark, run_id, all_test_results)
    print(f"Wrote {n} result rows")
except Exception as e:
    print(f"audit write FAILED: {e}")